# Model Risk - Varational Autoencoder (VAE)

A VAE is a kind of neural network that **learns a distribution** over a high-dimensional space. After training, you can **sample** new data points that look like the training data.

In [187]:
%load_ext autoreload
%autoreload 2

from garch import fit_ar_garch

import numpy as np
import pandas as pd

from arch import arch_model

import torch
import torch.nn as nn # The neural-network building blocks, such as Linear and ReLU.
import torch.nn.functional as F # MSE
from torch.utils.data import DataLoader, TensorDataset

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Basics

### 1.1 Load data

In [ ]:
# reset_index: Setzt den Index neu zurück; 
# Ohne diesen Schritt würden die alten Indizes (1, 2) erhalten bleiben
# drop=True sorgt dafür, dass der alte Index nicht als zusätzliche Spalte gespeichert wird
returns = pd.read_csv("/Users/andre/Documents/modelrisk/data/qrm2025_returns.csv")
returns = returns.iloc[1:].reset_index(drop=True) # reset_index: Setzt den Index neu zurück; 
returns.head()

,STOXX_EU_600,DOWJONES_INDUSTRIALS,MSCI_EM,S&P_U.S._TREASURY_BOND,S&P_GSCI_Commodity
0,0.021227,0.014988,0.015312,0.001415,0.025407
1,-0.001757,-0.001128,0.010706,0.002608,0.000994
2,-0.000513,0.000798,0.006397,-0.002013,0.017457
3,-0.003961,0.003267,-0.007171,-0.000337,-0.009653
4,0.004198,0.001069,0.001962,0.001275,-0.000069


### 1.2 Configuration

In [160]:
WINDOW = 500
N_FORECAST = len(returns) - WINDOW
N_SIM = 10000

ALPHA = 0.05, 0.01 # Signifikanzniveaus für VaR-Berechnung

SCALE = 1000 # scale factor for returns to improve GARCH convergence

SEED = 42

weights = np.ones(len(returns.columns)) / len(returns.columns)

N_FORECAST, weights

(1846, array([0.2, 0.2, 0.2, 0.2, 0.2]))

In [8]:
torch.manual_seed(SEED)

## 2. Rolling window

In [121]:
# Initializing DataFrames to store VaR results and realized returns for the forecast period
fcst_idx = returns.index[WINDOW: WINDOW + N_FORECAST] 

# var5 = pd.DataFrame(index=fcst_idx, columns=["VAE_model1", "VAE_model1"], dtype=float)
var_5 = pd.Series(index=fcst_idx, dtype=float, name='VaR_5%')

realized = pd.Series(index=fcst_idx, dtype=float, name='realized')


In [ ]:
for t in range(N_FORECAST): # Die Variable t läuft von 0 bis N_FORECAST-1; Für jede Iteration wird eine neue Prognose berechnet

    # --- Returns for the rolling window ------------------------------------------------------
    window_returns = returns.iloc[t:t+WINDOW]

    # --- Fit GARCH models to the rolling window ------------------------------------------------------
    garch_fits = {c: fit_ar_garch(returns_insample[c], garch_order=(1, 1), dist="t", horizon=1, scale=SCALE) for c in returns.columns}

    # --- Extract standardized residuals and build matrix for further calculation ------------------------------------------------------
    # AR(1) leaves a NaN in the first row
    matrix_residuals_std = np.array([f["residuals_std"] for f in garch_fits.values()]).T

    # Drop first row of NaN values
    mask = ~np.isnan(matrix_residuals_std).any(axis=1)
    matrix_residuals_std = matrix_residuals_std[mask]

    # --- VAE model ------------------------------------------------------





## SANDBOX: claude_vae_simple

### Database

In [88]:
# In-sample data: ersten 500 Beobachtungen
returns_insample = returns[:WINDOW]
returns_insample

,STOXX_EU_600,DOWJONES_INDUSTRIALS,MSCI_EM,S&P_U.S._TREASURY_BOND,S&P_GSCI_Commodity
0,0.021227,0.014988,0.015312,0.001415,0.025407
1,-0.001757,-0.001128,0.010706,0.002608,0.000994
2,-0.000513,0.000798,0.006397,-0.002013,0.017457
3,-0.003961,0.003267,-0.007171,-0.000337,-0.009653
4,0.004198,0.001069,0.001962,0.001275,-0.000069
...,...,...,...,...,...
495,0.043870,0.025929,0.028429,-0.000084,0.013701
496,0.006797,0.003794,0.008059,-0.001108,0.013417
497,0.046435,0.042415,0.020728,-0.002670,0.006942
498,-0.005980,-0.002130,0.033185,-0.000458,-0.005439


### Rolling window

#### Fitting garch models

In [67]:
returns_insample.columns

Index(['STOXX_EU_600', 'DOWJONES_INDUSTRIALS', 'MSCI_EM',
       'S&P_U.S._TREASURY_BOND', 'S&P_GSCI_Commodity'],
      dtype='object')

In [71]:
garch_fits = {c: fit_ar_garch(returns_insample[c], garch_order=(1, 1), dist="t", horizon=1, scale=SCALE) for c in returns.columns}

In [89]:
garch_fits.values()

dict_values([{'residuals_std': 0           NaN
1     -0.286642
2     -0.088718
3     -0.413051
4      0.357383
         ...   
495    2.122617
496    0.184548
497    2.067581
498   -0.349821
499    0.254444
Name: std_resid, Length: 500, dtype: float64, 'mu': np.float64(0.0008455634719929081), 'sigma': np.float64(0.02279250113659059), 'nu': np.float64(8.626029832333698)}, {'residuals_std': 0           NaN
1     -0.233827
2     -0.058078
3      0.266868
4     -0.013001
         ...   
495    1.736724
496    0.196125
497    2.732940
498   -0.111552
499   -0.068714
Name: std_resid, Length: 500, dtype: float64, 'mu': np.float64(0.0012434291712281793), 'sigma': np.float64(0.018154027286501118), 'nu': np.float64(4.466810226781494)}, {'residuals_std': 0           NaN
1      0.651900
2      0.335536
3     -0.969868
4      0.316498
         ...   
495    1.985768
496    0.048566
497    1.112749
498    1.681265
499   -0.369560
Name: std_resid, Length: 500, dtype: float64, 'mu': np.float64(0.00101

#### AR(1) leaves a NaN in the first row (no lag to use); drop it.

 AR(1) leaves a NaN in the first row (no lag to use) of the aggregated **matrix_residuals_std**.
    Dies liegt am AR(1)-Teil. Bei **mean="AR", lags=1** verliert das Modelle die erste Beobachtung, weil für den 
    ersten Zeitpunkt kein Lag existiert.

Aus den 5 Assets weden die standardisierten Residuen spaltenweise in einer einzigen Matrix zusammenführen, sodass jede Spalte ein Asset und jede Zeile eine Beobachtung des Rolling Windows ist

--> Shape(500,5): Anzahl der Zeitpunkte im Window x Anzahl Assets

Implikation: Bei einer **Matrix mit Shape (500,5)** werden die Operationen spaltenweise durchgeführt, weil jede Spalte eine Variable (hier: ein Asset) repräsentiert.

##### 1. Variante

In [92]:
returns_insample.columns

Index(['STOXX_EU_600', 'DOWJONES_INDUSTRIALS', 'MSCI_EM',
       'S&P_U.S._TREASURY_BOND', 'S&P_GSCI_Commodity'],
      dtype='object')

In [96]:
test_matrix = pd.concat(
    [garch_fits[a]["residuals_std"] for a in returns_insample.columns],
    axis=1
)
test_matrix.columns = returns.columns

In [97]:
test_matrix

,STOXX_EU_600,DOWJONES_INDUSTRIALS,MSCI_EM,S&P_U.S._TREASURY_BOND,S&P_GSCI_Commodity
0,NaN,NaN,NaN,NaN,NaN
1,-0.286642,-0.233827,0.651900,1.244948,-0.060640
2,-0.088718,-0.058078,0.335536,-1.057055,1.333490
3,-0.413051,0.266868,-0.969868,-0.340011,-0.855136
4,0.357383,-0.013001,0.316498,0.492587,-0.044871
...,...,...,...,...,...
495,2.122617,1.736724,1.985768,-0.219986,1.018073
496,0.184548,0.196125,0.048566,-0.646576,0.934302
497,2.067581,2.732940,1.112749,-1.425523,0.431632
498,-0.349821,-0.111552,1.681265,-0.388097,-0.517103


In [98]:
test_matrix.shape

(500, 5)

In [101]:
test_matrix = test_matrix.dropna()
test_matrix

,STOXX_EU_600,DOWJONES_INDUSTRIALS,MSCI_EM,S&P_U.S._TREASURY_BOND,S&P_GSCI_Commodity
1,-0.286642,-0.233827,0.651900,1.244948,-0.060640
2,-0.088718,-0.058078,0.335536,-1.057055,1.333490
3,-0.413051,0.266868,-0.969868,-0.340011,-0.855136
4,0.357383,-0.013001,0.316498,0.492587,-0.044871
5,1.277980,0.449320,1.125339,-0.152593,-0.410423
...,...,...,...,...,...
495,2.122617,1.736724,1.985768,-0.219986,1.018073
496,0.184548,0.196125,0.048566,-0.646576,0.934302
497,2.067581,2.732940,1.112749,-1.425523,0.431632
498,-0.349821,-0.111552,1.681265,-0.388097,-0.517103


##### 2. Variante - gewünschte Lösung

In [163]:
# AR(1) leaves a NaN in the first row (no lag to use); drop it.
# ohne .T: shape(5,500) -> 5 Assets (rows) x 500 Zeitpunkte (columns)
# mit .T: shape(500, 5) -> Anzahl Zeitpunkte (rows) x Anzahl Assets (columns)
matrix_residuals_std = np.array([f["residuals_std"] for f in garch_fits.values()]).T
matrix_residuals_std

array([[        nan,         nan,         nan,         nan,         nan],
       [-0.28664158, -0.23382707,  0.65189964,  1.24494751, -0.06064044],
       [-0.08871803, -0.05807786,  0.33553551, -1.05705495,  1.33349   ],
       ...,
       [ 2.06758059,  2.73294003,  1.11274862, -1.42552311,  0.43163197],
       [-0.34982097, -0.11155154,  1.68126459, -0.38809703, -0.51710302],
       [ 0.25444405, -0.06871373, -0.36955983,  1.03546767,  0.42245071]],
      shape=(500, 5))

Entferne alle Zeilen, die mindestens ein NaN enthalten

In [164]:
# mask for non-NaN values
mask = ~np.isnan(matrix_residuals_std).any(axis=1)
mask

array([False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [165]:
z = matrix_residuals_std[mask]
z

array([[-0.28664158, -0.23382707,  0.65189964,  1.24494751, -0.06064044],
       [-0.08871803, -0.05807786,  0.33553551, -1.05705495,  1.33349   ],
       [-0.41305072,  0.26686772, -0.96986848, -0.34001093, -0.85513563],
       ...,
       [ 2.06758059,  2.73294003,  1.11274862, -1.42552311,  0.43163197],
       [-0.34982097, -0.11155154,  1.68126459, -0.38809703, -0.51710302],
       [ 0.25444405, -0.06871373, -0.36955983,  1.03546767,  0.42245071]],
      shape=(499, 5))

### Building VAE model

Two main parts of the VAE model:
* **Encoder:** produces TWO outputs from x: mu(x) and log_var(x).
* **Decoder:** takes a 2-dim latent z and reconstructs a 5-dim vector.

Why log_var, not var? A variance must be positive, but a NN output is unconstrained. We let the network output log_var (any real number), then recover sigma = exp(0.5 * log_var) > 0. Standard trick.

--------------------------------------------------------------------------------
A **generative model** like a VAE is a neural network that, after training, can **sample** new data points that look like the training data.

To make this concrete: imagine your training data consists of 500 five-asset return vectors (one row per trading day). A good generative model trained on this data should be able to:

* output 10,000 brand new five-asset return vectors,
* such that the new vectors have the same per-asset mean and standard
  deviation as the real data,
* the same correlations between assets,
* the same fat-tailed behavior,
* and so on for any statistical property that mattered in training.


Steps:  standardized   ──▶  DEPENDENCE  ──▶  10,000 simulated
         residuals          MODEL              standardized residuals
                            (the VAE)

--------------------------------------------------------------------------------
The **standardized residuals** look approximately like:

* mean 0 in each column (one column per asset),
* std 1 in each column,
* still correlated across columns (positive correlation between equity
  indices, slight negative between stocks and treasury bonds, etc.).

The VAE has to capture exactly those cross-asset correlations -- and,
crucially, the joint tail behavior.


--------------------------------------------------------------------------------
architecture in pseudo-diagram form:

```
x (5-dim)  ──▶  Linear(5, 16)  ──▶  tanh  ──▶  hidden (16-dim)
                                                     │
                                          ┌──────────┴──────────┐
                                          ▼                     ▼
                                    Linear(16, 2)         Linear(16, 2)
                                    = mu                  = log_var
                                          │                     │
                                          └──── sample z ───────┘
                                                z = mu + sigma * eps
                                                eps ~ N(0, 1)
                                                     │
                                                     ▼
                                              z (2-dim)
                                                     │
                                                     ▼
                                          Linear(2, 16)  ──▶  tanh  ──▶  Linear(16, 5)
                                                                              │
                                                                              ▼
                                                                         x_hat (5-dim)
```

--------------------------------------------------------------------------------

So a VAE is shaped like an **autoencoder** (input → smaller latent → back to input) BUT with two differences from a plain autoencoder:

* Difference 1: the **latent is a distribution**, not a point
* Difference 2: the loss has a **second term-- the KL divergence**

**Difference 1: the latent is a distribution, not a point**

A plain *autoencoder* maps each input x to a **single point z** in latent space. A *VAE* maps each x to a small **Gaussian distribution N(mu(x), sigma(x)²)** over latent space. Then, to actually feed something to the decoder, we **sample** z from that distribution.

This sampling step needs the **reparameterization trick** to keep gradients flowing:

    z = mu(x) + sigma(x) * eps,    eps ~ N(0, 1)

The randomness is in `eps` (external to the network). The parameters mu(x) and sigma(x) are differentiable network outputs. Gradients of the loss with respect to mu and sigma are well-defined.

Without this trick, the random sampling would break autograd and the network couldn't train.


**Difference 2: the loss has a second term -- the KL divergence**

The loss is

    Loss  =  Reconstruction(x, x_hat)  +  beta * KL( N(mu, sigma²) || N(0, I) )

The first term is just MSE -- the same as in the MLP. It says "the decoded x_hat should look like the input x".

The **KL term** is new and is the magic. It compares the **encoder's output distribution N(mu(x), sigma²(x))** to the **Gaussian standard normal N(0, I)**. **The closer they are, the smaller the KL**. Loosely:

    KL ≈ 0    ⟺    the encoder maps every x to roughly N(0, I)
    KL >> 0   ⟺    the encoder produces something wildly non-normal

We MINIMIZE the loss, so the network is pushed toward "encoder output looks like N(0, I)".



**Why this is enough to make the model generative ?**

After training, the **encoder has been pushed (by the KL term) to map real data to a region of latent space that overlaps with N(0, I)**. To generate new data, we simply:

1. Sample `z ~ N(0, I)` (the prior),
2. Pass z through the decoder,
3. Get a synthetic x.

Because real data was mapped INTO the N(0, I) region, and the decoder learned to reconstruct real data from points in that region, drawing fresh N(0, I) samples and decoding them produces **outputs that look like the training data**.



**Beta**

The coefficient `beta` weights the KL term. `beta = 1` is the textbook VAE. The exposé's baseline uses `beta = 2`. Higher beta means more pressure on the latent to look Gaussian, at the cost of worse reconstruction.

Why does the exposé prefer `beta = 2` instead of `beta = 1`? It is a deliberate tradeoff: a more strongly-regularized latent makes the generative sampling step more reliable but produces blurrier samples.
The exposé's **L2 sensitivity experiment** asks exactly this question -- whether beta=1 or beta=2 or beta=4 produces lower model risk in practice. We use beta=1 here for clarity. The thesis answers the question.

#### Configuration of VAE model

VAE model **komprimiert** Vektor mit standardisierten Residuen auf eine Dimension (**Latent space**) mit der Größe=2. Das Model zwingt den Marktzustand in eine Struktur mit der Dimension=2.

Der VAE lernt:
* Abhängigkeiten
* Verteilung
* Non-linearity
alles in 2 latenten Faktoren/Dimensionen.

Diese 2 Dimensionen können, z.B. sein:
* Marktstress
* Volatility clusters

In [128]:
INPUT_DIM = returns_insample.shape[1]
HIDDEN_DIM = 16
LATENT_DIM = 2 # latent space

# Regularizaiton params
# beta weights the KL term
# higher beta means more pressure on the latent to look Gaussian, at the cost of worse reconstruction.
BETA = 1.0 

INPUT_DIM

5

#### VAE class

In [129]:
torch.manual_seed(SEED)

In [149]:
class VAE_simple(nn.Module): # Definition eines NN in PyTorch; nn.Module: Basis-Klasse für alle Modelle
    def __init__(self): # Aufbau des Netzes: hier wird die Architektur definiert
        super().__init__()

        # Encoder: x (Input data) -> hidden
        self.encoder = nn.Sequential(
            nn.Linear(INPUT_DIM, HIDDEN_DIM), # 5 Asset-Informationen werden zu einer gemeinsamen Repräsentation gemischt
            nn.Tanh(), # Aktivierungsfunktion: Werte werden auf [-1, 1] begrenzt -> führt non-linearity ein, da ohne Aktivierung wäre nur alles linear
        )

        # VAE-specific part: LATENT SPACE
        # Hidden layer wird in eine Verteilung (mu, sigma bzw. log_var) zerlegt, nicht in einen festen Wert
        # Warum logvar? stabiler numerisch, verhindert negative Varianz, leichter zu optimieren
        # Warum mu und sigma? 
            # Es wird nicht modelliert: x -> z
            # sondern (Kern des VAE): x → (μ(x), σ²(x))
        self.encoder_mu = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.encoder_logvar = nn.Linear(HIDDEN_DIM, LATENT_DIM)


        # Decoder: z -> hidden -> x_hat
        self.decoder = nn.Sequential(
            # komprimierter "Marktzustand" aus dem latent space wird in eine größere Repräsentation expandiert/entfaltet
            # --> Du “entfaltest” den latenten Faktorraum zurück in ein reichhaltigeres Feature-Space.
            nn.Linear(LATENT_DIM, HIDDEN_DIM), 
            
            # Aktivierungsfunktion
            nn.Tanh(),

            # Output: 5 Werte, d.h. rekonstruierte standardisierte Residuen x_hat
            nn.Linear(HIDDEN_DIM, INPUT_DIM),
        )


    # Funktion für den Encoder
    def encode(self, x):
        # AE: x → h → z → x_hat
        # VAE: x → h → (μ, σ²) → z ~ N(μ, σ²) → x_hat
        # x: Input data mit der shape (batch_size, 5)
        h = self.encoder(x) # Ergebnis: shape(batch_size, HIDDEN_DIM) -> jetzt: abstrakte Marktrepräsentation
        return self.encoder_mu(h), self.encoder_logvar(h)

    # Funktion für den Decoder
    def decode(self, z):
        return self.decoder(z)

    
    # Reparameterization trick
    # Aus dem Encoder hat man:
        # mu: Mittelwert des latent space
        # logvar
        # -> Daraus sollen neue Werte gesampelt werden z ~ N(mu, variance)
    # Problematik:
        # Sampling ist nicht differenzierbar
        # Backpropagation kann nicht durch Zufall laufen
        # Gradient stoppt → Training bricht konzeptionell
    # Idee des Reparameterization trick
        # statt direkt zu sampeln z ~ N(mu, variance)
        # schreibt man: z = mu + sigma * epsilon mit eps ~ N(0,1)
    # Warum funktioniert das ?
        # Trick ist: Die Zufälligkeit ist jetzt außerhalb des NN
        # mu und sigma sind deterministisch vom Modell
        # eps ist externens, zufälliges Rauschen
    # Jetzt mit Output: 
        # Verlust → z → μ, σ
        # eps wird wie eine Konstante im Gradient behandelt
        # Implikation: Das NN bleibt dadurch vollständig differenzierbar
    # Intuition:
        # ohne Trick: "Ich ziehe zufällig einen Punkt" -> nicht ableitbar
        # mit Trick: "Ich nehme einen festen Zufallswert eps und verschiebe/skaliere ihn" -> vollständig ableitbar
    # Im Kontext des Finanzmarktes
        # Mu: erwarteter latenter Marktzustand
        # sigma: Unsicherheit des Marktzustandes
        # eps: zufällige Marktbewegung
        # Ergebnis -> z: Marktregime + zufällige Schockkomponente
    def reparameterize(self, mu, logvar):
        """
        The reparameterization trick.

        We want to sample z ~ N(mu, sigma^2). Naively writing
            z = torch.normal(mu, sigma)
        breaks autograd: there is no gradient through a random draw.
        Instead we write
            z = mu + sigma * eps,   eps ~ N(0, 1)
        which IS differentiable in mu and sigma because the randomness
        is now external (in eps).
        """
        sigma = torch.exp(0.5 * logvar)
        eps = torch.randn_like(sigma) # Erzeugen von Zufallsrauschen epsilon -> dieser Part ist die einzige echte Zufälligkeit
        return mu + sigma * eps # Reparameterization z = mu + sigma * eps,   eps ~ N(0, 1)


    # Forward pass
    def forward(self, x): # So wird ein Input x durch das NN verarbeitet
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar) # Reparameterization trick

        # Was wird zurückgegeben?
            # Rekonstruktion x_hat: geschätzte Residuen für 5 Assets
            # mu udn logvar
        return self.decode(z), mu, logvar


    # Sampling
    # Ziel: Erzeugung n neuer synthetischer Residuen-Vektoren
        # jeder Vektor hat 5 Dim
        # sollen wie echte Daten aussehn, aber neu sein
    @torch.no_grad() # Bedeutung: Beim Sampling wird kein Gradient berechnet; Why? hier wird nicht trainienrt, es werden nur Daten generiert, spart Speicher und Rechenzeit
    def sampling(self, n):
        """
        Generate n synthetic residual vectors. The exposé's recipe:
        draw z ~ N(0, I), pass through the decoder.
        """
        z = torch.randn(n, LATENT_DIM)
        return self.decode(z)


#### Initialization of VAE model

In [150]:
# VAE_simple wird instanziiert
# alle Layer werden erzeugt: Encoder, Decoder, mu, logvar
# Ergebnis: neues VAE model im Speicher/Storage
model = VAE_simple()

In [151]:
# Wie komplex ist mein generatives Modell eigentlich?
n_params = sum(p.numel() for p in model.parameters())
print(f"Step 3.  Built VAE  (input_dim={INPUT_DIM}, hidden={HIDDEN_DIM}, "
      f"latent_dim={LATENT_DIM}, beta={BETA})")
print(f"         trainable parameters: {n_params}\n")

Step 3.  Built VAE  (input_dim=5, hidden=16, latent_dim=2, beta=1.0)
         trainable parameters: 297



#### Define the loss = reconstruction + beta * KL

The "evidence lower bound" (ELBO) for a VAE consists of two parts:
* reconstruction term:  encourage the decoded x_hat to be close to x. With a Gaussian likelihood and fixed variance, this reduces to mean squared error.
* KL divergence term:   encourage q(z|x) = N(mu, sigma^2) to look like the standard normal prior N(0, I). For diagonal Gaussians there is a closed form: KL = 0.5 * sum( mu^2 + sigma^2 - log(sigma^2) - 1 )

The beta coefficient weights these. beta = 1 is the textbook VAE. beta > 1 makes the latent "more Gaussian" at the cost of reconstruction fidelity. The exposé baseline uses beta = 2.

Loss function des VAE, namely evidence lower bound (**ELBO**). Die loss function besteht aus 2 Parts und ist entscheidend dafür, dass dein Modell sowohl gute Rekonstruktionen lernt als auch eine sinnvolle latente Struktur aufbaut.
* **reconstruction term**: Kann ich die Daten gut nachbauen?
* **KL divergence**: Ist der latent space normalverteilt?

In [152]:
# x: echte Daten, hier standardized_residuals
# x_hat: Rekonstruktion vom Decoder
# Beta: Gewichtung des KL-Terms
def vae_loss(x, x_hat, mu, log_var, beta=BETA):
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.size(0) # Rekonstruktionsfehler wie MSE -> Wie gut rekonstruiert der Decoder die echten Daten?
    kl    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0) # Ziel des KL-Terms: der latent space soll "normal ausshen", d.h. SNV aufweisen
    return recon + beta * kl, recon, kl

#### Train the VAE model
hier: VAE passt seine Gewichte so an, dass die Residuen gut rekonstruiert werden und gleichzeitig einen strukturierten latent space lernt (latent space hat SNV).

In [166]:
Z_tensor = torch.from_numpy(z).float()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Step 5.  Training (full-batch, 500 epochs):")
for epoch in range(1, 501):
    model.train()
    optimizer.zero_grad()
    x_hat, mu, logvar = model(Z_tensor)
    loss, recon, kl    = vae_loss(Z_tensor, x_hat, mu, logvar)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch:>3}   total = {loss.item():.4f}   "
              f"recon = {recon.item():.4f}   KL = {kl.item():.4f}")
print()

Step 5.  Training (full-batch, 500 epochs):
epoch 100   total = 3.1233   recon = 2.0027   KL = 1.1206
epoch 200   total = 3.1213   recon = 1.9839   KL = 1.1374
epoch 300   total = 3.0868   recon = 1.9355   KL = 1.1513
epoch 400   total = 3.2298   recon = 2.0849   KL = 1.1449
epoch 500   total = 3.2223   recon = 2.0894   KL = 1.1329



In [159]:
Z_tensor

tensor([[-0.2866, -0.2338,  0.6519,  1.2449, -0.0606],
        [-0.0887, -0.0581,  0.3355, -1.0571,  1.3335],
        [-0.4131,  0.2669, -0.9699, -0.3400, -0.8551],
        ...,
        [ 2.0676,  2.7329,  1.1127, -1.4255,  0.4316],
        [-0.3498, -0.1116,  1.6813, -0.3881, -0.5171],
        [ 0.2544, -0.0687, -0.3696,  1.0355,  0.4225]])

#### Sampling of new synthetic data

In [167]:
model.eval()
z_sim = model.sampling(N_SIM).numpy()

print("Step 6.  Diagnostics for the SAMPLED residuals")
print(f"  empirical mean:     {z.mean(axis=0).round(2)}")
print(f"  simulated mean:     {z_sim.mean(axis=0).round(2)}")
print(f"  empirical std:      {z.std(axis=0).round(2)}")
print(f"  simulated std:      {z_sim.std(axis=0).round(2)}   "
      f"<-- watch this")
print()
print("  empirical correlation matrix:")
print(np.corrcoef(z.T).round(2))
print("  simulated correlation matrix:")
print(np.corrcoef(z_sim.T).round(2))
print()

Step 6.  Diagnostics for the SAMPLED residuals
  empirical mean:     [-0.04 -0.09 -0.05 -0.   -0.05]
  simulated mean:     [-0.04 -0.09 -0.05 -0.01 -0.05]
  empirical std:      [0.99 0.98 1.   1.   1.  ]
  simulated std:      [0.82 0.78 0.79 0.71 0.72]   <-- watch this

  empirical correlation matrix:
[[ 1.    0.72  0.75 -0.39  0.6 ]
 [ 0.72  1.    0.58 -0.56  0.55]
 [ 0.75  0.58  1.   -0.3   0.57]
 [-0.39 -0.56 -0.3   1.   -0.35]
 [ 0.6   0.55  0.57 -0.35  1.  ]]
  simulated correlation matrix:
[[ 1.    0.96  0.99 -0.68  0.99]
 [ 0.96  1.    0.91 -0.86  0.95]
 [ 0.99  0.91  1.   -0.58  0.99]
 [-0.68 -0.86 -0.58  1.   -0.66]
 [ 0.99  0.95  0.99 -0.66  1.  ]]



In [171]:
z_sim

array([[ 1.1923436 ,  0.9424324 ,  1.1075889 , -0.59628016,  0.95452994],
       [-1.2403482 , -1.2548614 , -1.1970242 ,  0.83016866, -1.1650192 ],
       [-0.51914406, -0.8555148 , -0.32523927,  1.0441686 , -0.39215913],
       ...,
       [ 0.5717821 ,  0.1825689 ,  0.69218934,  0.46456334,  0.54994863],
       [ 0.01544815, -0.0462979 ,  0.05347638,  0.06597598,  0.06254267],
       [ 0.5195209 ,  0.47751412,  0.45967916, -0.5038602 ,  0.458266  ]],
      shape=(10000, 5), dtype=float32)

### Compute one-step VAR

In [168]:
garch_fits.values()

dict_values([{'residuals_std': 0           NaN
1     -0.286642
2     -0.088718
3     -0.413051
4      0.357383
         ...   
495    2.122617
496    0.184548
497    2.067581
498   -0.349821
499    0.254444
Name: std_resid, Length: 500, dtype: float64, 'mu': np.float64(0.0008455634719929081), 'sigma': np.float64(0.02279250113659059), 'nu': np.float64(8.626029832333698)}, {'residuals_std': 0           NaN
1     -0.233827
2     -0.058078
3      0.266868
4     -0.013001
         ...   
495    1.736724
496    0.196125
497    2.732940
498   -0.111552
499   -0.068714
Name: std_resid, Length: 500, dtype: float64, 'mu': np.float64(0.0012434291712281793), 'sigma': np.float64(0.018154027286501118), 'nu': np.float64(4.466810226781494)}, {'residuals_std': 0           NaN
1      0.651900
2      0.335536
3     -0.969868
4      0.316498
         ...   
495    1.985768
496    0.048566
497    1.112749
498    1.681265
499   -0.369560
Name: std_resid, Length: 500, dtype: float64, 'mu': np.float64(0.00101

In [170]:
# --- Simulated returns: r_sim = mu + sigma * z_sim ------------------------------------------------------
vec_mu = np.column_stack([f["mu"] for f in garch_fits.values()])
vec_sigma = np.column_stack([f["sigma"] for f in garch_fits.values()])

vec_mu, vec_sigma

(array([[0.00084556, 0.00124343, 0.00101276, 0.00013327, 0.00107188]]),
 array([[0.0227925 , 0.01815403, 0.01694478, 0.0021781 , 0.0122642 ]]))

In [174]:
r_sim = vec_mu + vec_sigma * z_sim
r_sim

array([[ 0.02802206,  0.01835237,  0.01978061, -0.00116548,  0.01277842],
       [-0.02742507, -0.02153736, -0.01927056,  0.00194146, -0.01321615],
       [-0.01098703, -0.01428761, -0.00449835,  0.00240757, -0.00373764],
       ...,
       [ 0.01387791,  0.00455779,  0.01274176,  0.00114514,  0.00781656],
       [ 0.00119767,  0.00040294,  0.0019189 ,  0.00027697,  0.00183891],
       [ 0.01268674,  0.00991223,  0.00880192, -0.00096418,  0.00669214]],
      shape=(10000, 5))

In [175]:
pf_sim = np.dot(r_sim, weights)
pf_sim

array([ 0.0155536 , -0.01590154, -0.00622061, ...,  0.00802783,
        0.00112708,  0.00742577], shape=(10000,))

In [176]:
# Risk measure forecasts
np.percentile(pf_sim, ALPHA[0]) # VaR at 5% level

np.float64(-0.03467946409976435)

## SANDBOX 2: claude_vae_complex 

#### Configuration

In [186]:
INPUT_DIM = returns_insample.shape[1]
# HIDDEN_DIM = 16
HIDDEN_DIMS = (64, 64)
LATENT_DIM = 3 # latent space

# Regularizaiton params
# beta weights the KL term
# higher beta means more pressure on the latent to look Gaussian, at the cost of worse reconstruction.
BATCH_SIZE     = 32                
MAX_EPOCHS     = 200

BETA = 2.0

# 
LEARNING_RATE  = 1e-3



INPUT_DIM

5

#### VAE class

In [184]:
torch.manual_seed(SEED)

In [183]:
class VAE_model_complex(nn.Module): # Definition eines NN in PyTorch; nn.Module: Basis-Klasse für alle Modelle
    """
    Exposé baseline architecture (page 2): 
     - 2 hidden x 64 tanh, 
     - latent=3,
     - beta=2, 
     - MSE reconstruction
    """
    def __init__(self, input_dim, hidden_dims=HIDDEN_DIMS, latent_dim=LATENT_DIM, beta=BETA): # Aufbau des Netzes: hier wird die Architektur definiert
        super().__init__()
        
        self.latent_dim = latent_dim
        self.beta       = beta

        # Encoder: x (Input data) -> 64 -> 64
        # Pipeline: Linear -> Tanh -> Linear -> Tanh
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dims[0]), # 5 Asset-Informationen werden zu einer gemeinsamen Repräsentation gemischt
            nn.Tanh(), # Aktivierungsfunktion: Werte werden auf [-1, 1] begrenzt -> führt non-linearity ein, da ohne Aktivierung wäre nur alles linear
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.Tanh()
        )

        # VAE-specific part: LATENT SPACE
        # Hidden layer wird in eine Verteilung (mu, sigma bzw. log_var) zerlegt, nicht in einen festen Wert
        # Warum logvar? stabiler numerisch, verhindert negative Varianz, leichter zu optimieren
        # Warum mu und sigma? 
            # Es wird nicht modelliert: x -> z
            # sondern (Kern des VAE): x → (μ(x), σ²(x))
        self.encoder_mu = nn.Linear(hidden_dims[1], latent_dim)
        self.encoder_logvar = nn.Linear(hidden_dims[1], latent_dim)


        # Decoder: z -> hidden -> x_hat
        # Decoder mirrors encoder
        self.decoder = nn.Sequential(
            # komprimierter "Marktzustand" aus dem latent space wird in eine größere Repräsentation expandiert/entfaltet
            # --> Du “entfaltest” den latenten Faktorraum zurück in ein reichhaltigeres Feature-Space.
            nn.Linear(latent_dim, hidden_dims[1]),
            nn.Tanh(), # Aktivierungsfunktion
            nn.Linear(hidden_dims[1], hidden_dims[0]),
            nn.Tanh(), # Aktivierungsfunktion
            nn.Linear(hidden_dims[0], input_dim) # Output layer: 5 Werte, d.h. rekonstruierte standardisierte Residuen x_hat
        )




    # Funktion für den Encoder
    def encode(self, x):
        # AE: x → h → z → x_hat
        # VAE: x → h → (μ, σ²) → z ~ N(μ, σ²) → x_hat
        # x: Input data mit der shape (batch_size, 5)
        h = self.encoder(x) # Ergebnis: shape(batch_size, HIDDEN_DIM) -> jetzt: abstrakte Marktrepräsentation
        return self.encoder_mu(h), self.encoder_logvar(h)

    # Funktion für den Decoder
    def decode(self, z):
        return self.decoder(z)

    
    # Reparameterization trick
    def reparameterize(self, mu, logvar):
        """
        The reparameterization trick.

        We want to sample z ~ N(mu, sigma^2). Naively writing
            z = torch.normal(mu, sigma)
        breaks autograd: there is no gradient through a random draw.
        Instead we write
            z = mu + sigma * eps,   eps ~ N(0, 1)
        which IS differentiable in mu and sigma because the randomness
        is now external (in eps).
        """
        sigma = torch.exp(0.5 * logvar)
        eps = torch.randn_like(sigma) # Erzeugen von Zufallsrauschen epsilon -> dieser Part ist die einzige echte Zufälligkeit
        return mu + sigma * eps # Reparameterization z = mu + sigma * eps,   eps ~ N(0, 1)


    # Forward pass
    def forward(self, x): # So wird ein Input x durch das NN verarbeitet
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar) # Reparameterization trick

        # Was wird zurückgegeben?
            # Rekonstruktion x_hat: geschätzte Residuen für 5 Assets
            # mu udn logvar
        return self.decode(z), mu, logvar


    # Sampling
    @torch.no_grad() # Bedeutung: Beim Sampling wird kein Gradient berechnet; Why? hier wird nicht trainienrt, es werden nur Daten generiert, spart Speicher und Rechenzeit
    def sampling(self, n):
        """
        Generate n synthetic residual vectors. The exposé's recipe:
        draw z ~ N(0, I), pass through the decoder.
        """
        z = torch.randn(n, LATENT_DIM)
        return self.decode(z)

In [185]:
# x: echte Daten, hier standardized_residuals
# x_hat: Rekonstruktion vom Decoder
# Beta: Gewichtung des KL-Terms
def vae_loss(x, x_hat, mu, log_var, beta):
    recon = F.mse_loss(x_hat, x, reduction="sum") / x.size(0)
    kl    = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    return recon + beta * kl

In [188]:
def vae_loss_components(x, x_hat, mu, lv, beta=1.0):
    """
    Return (total, recon, kl) so we can monitor them separately.
    Assumes the same per-sample reductions as the original vae_loss.
    """
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl    = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
    total = recon + beta * kl
    return total, recon, kl

#### VAE training loop

##### with train/val

In [193]:
def VAE_train(model, z, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, lr=LEARNING_RATE, val_frac=0.15, verbose=True):
    """
    Train the VAE with mini-batch SGD.

    Chronological train/val split BEFORE shuffling into batches
    (never let future information leak into training).
    Within the training portion we DO shuffle batches each epoch.

    Logs reconstruction and KL separately on both train and val so
    posterior collapse is visible (KL → 0 while recon still improves).



    Parameters
    ----------
    val_frac : float in [0, 1)
        Fraction of the (chronologically last) data used for validation.
        Set to 0.0 to train on all data without a validation split.
    """
    n_val = int(val_frac * z.shape[0])
    if val_frac > 0.0:
        n_val = max(1, n_val)
        z_tr = torch.from_numpy(z[:-n_val]).float()
        z_va = torch.from_numpy(z[-n_val:]).float()
    else:
        z_tr = torch.from_numpy(z).float()
        z_va = None

    # mini-batch loader
    loader = DataLoader(TensorDataset(z_tr),
                            batch_size=batch_size, shuffle=True)
    
    # Optimizer
    opt    = torch.optim.Adam(model.parameters(), lr=lr)


    history = {"train_total": [], "train_recon": [], "train_kl": []}
    if z_va is not None:
        history.update({"val_total": [], "val_recon": [], "val_kl": []})

    for epoch in range(1, max_epochs + 1):
        # ---- Train ----
        model.train()

        sum_tot = sum_rec = sum_kl = 0.0
        n_seen  = 0

        for (xb,) in loader:
            opt.zero_grad()
            x_hat, mu, lv = model(xb)
            total, recon, kl = vae_loss_components(xb, x_hat, mu, lv,
                                                   beta=model.beta)
            total.backward()
            opt.step()

            bs = xb.size(0)
            sum_tot += total.item() * bs
            sum_rec += recon.item() * bs
            sum_kl  += kl.item()    * bs
            n_seen  += bs

        tr_tot = sum_tot / n_seen
        tr_rec = sum_rec / n_seen
        tr_kl  = sum_kl  / n_seen
        history["train_total"].append(tr_tot)
        history["train_recon"].append(tr_rec)
        history["train_kl"].append(tr_kl)

        # ---- Validate ----
        if z_va is not None:
            model.eval()
            with torch.no_grad():
                x_hat_v, mu_v, lv_v = model(z_va)
                va_tot, va_rec, va_kl = vae_loss_components(
                    z_va, x_hat_v, mu_v, lv_v, beta=model.beta)
                va_tot, va_rec, va_kl = va_tot.item(), va_rec.item(), va_kl.item()
            history["val_total"].append(va_tot)
            history["val_recon"].append(va_rec)
            history["val_kl"].append(va_kl)

            if verbose and epoch % 20 == 0:
                print(f"  epoch {epoch:>3}   "
                      f"train: tot={tr_tot:.4f} rec={tr_rec:.4f} kl={tr_kl:.4f}   "
                      f"val: tot={va_tot:.4f} rec={va_rec:.4f} kl={va_kl:.4f}")
        else:
            if verbose and epoch % 20 == 0:
                print(f"  epoch {epoch:>3}   "
                      f"train: tot={tr_tot:.4f} rec={tr_rec:.4f} kl={tr_kl:.4f}   "
                      f"(no validation set)")
    return history

##### withouth train/val

In [194]:
def train_vae(model, z, batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
              lr=LEARNING_RATE, verbose=True):
    """
    Train the VAE with mini-batch SGD on the full dataset.

    Logs reconstruction and KL separately so posterior collapse is
    visible (KL → 0 while recon still improves).

    Returns
    -------
    history : dict of lists
        Per-epoch metrics: 'train_total', 'train_recon', 'train_kl'.
    """
    z_tr = torch.from_numpy(z).float()

    loader = DataLoader(TensorDataset(z_tr),
                        batch_size=batch_size, shuffle=True)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_total": [], "train_recon": [], "train_kl": []}

    for epoch in range(1, max_epochs + 1):
        model.train()
        sum_tot = sum_rec = sum_kl = 0.0
        n_seen  = 0
        for (xb,) in loader:
            opt.zero_grad()
            x_hat, mu, lv = model(xb)
            total, recon, kl = vae_loss_components(xb, x_hat, mu, lv,
                                                   beta=model.beta)
            total.backward()
            opt.step()

            bs = xb.size(0)
            sum_tot += total.item() * bs
            sum_rec += recon.item() * bs
            sum_kl  += kl.item()    * bs
            n_seen  += bs

        tr_tot = sum_tot / n_seen
        tr_rec = sum_rec / n_seen
        tr_kl  = sum_kl  / n_seen
        history["train_total"].append(tr_tot)
        history["train_recon"].append(tr_rec)
        history["train_kl"].append(tr_kl)

        if verbose and epoch % 20 == 0:
            print(f"  epoch {epoch:>3}   "
                  f"tot={tr_tot:.4f}  rec={tr_rec:.4f}  kl={tr_kl:.4f}")

    return history

### Main pipeline

#### Functions

##### AR-GARCH per asset

Fitting marginals

In [ ]:
def fit_ar_garch(returns, garch_order=(1, 1), lags=1, dist="t",
                 horizon=1, scale=100):
    """
    Fit AR(1)-GARCH(1,1) with Student-t innovations.

    Note: AR(1) leaves a NaN in the first row of the standardized residuals
    because there is no lag for the first observation.

    Returns
    -------
    dict with:
        residuals_std : standardized residuals (innovations)
        mu            : one-step-ahead mean forecast (original scale)
        sigma         : one-step-ahead vol forecast (original scale)
        nu            : Student-t degrees of freedom (or None)
    """
    from arch import arch_model

    p, q = garch_order
    res = arch_model(returns * scale, mean="AR", lags=lags,
                     vol="Garch", p=p, q=q, dist=dist
                     ).fit(disp="off", show_warning=False)

    nu            = res.params["nu"] if dist == "t" else None
    residuals_std = res.std_resid

    fc    = res.forecast(horizon=horizon)
    sigma = np.sqrt(fc.variance.values[-1, 0]) / scale
    mu    = fc.mean.iloc[-1, 0] / scale

    return {"residuals_std": residuals_std,
            "mu": mu, "sigma": sigma, "nu": nu}

Build residuals matrix

In [ ]:
def build_residuals_matrix(returns_window, scale=SCALE):
    """
    Fit AR(1)-GARCH(1,1)-t per asset on the rolling window.

    Returns
    -------
    resid : (T-1, n_assets) standardized residuals (NaN first row dropped)
    mu    : (n_assets,)     one-step-ahead mean forecasts
    sigma : (n_assets,)     one-step-ahead vol forecasts
    fits  : dict            per-asset fit dicts (for inspection)
    """
    fits = {c: fit_ar_garch(returns_window[c], garch_order=(1, 1),
                            dist="t", horizon=1, scale=scale)
            for c in returns_window.columns}

    resid = np.array([f["residuals_std"] for f in fits.values()]).T
    mask  = ~np.isnan(resid).any(axis=1)
    resid = resid[mask]

    mu    = np.array([f["mu"]    for f in fits.values()])
    sigma = np.array([f["sigma"] for f in fits.values()])
    return resid, mu, sigma, fits

Compute VaR from simulated residuals

In [ ]:
def compute_var(z_sim, mu, sigma, weights, alpha=(0.05, 0.01)):
    """
    Given simulated standardized residuals and the GARCH one-step ahead forecasts,
    produce the portfolio 95% and 99% VaR.
    """
    r_sim = mu + sigma * z_sim
    pf_sim = np.dot(r_sim, weights)
    var_5 = np.percentile(pf_sim, alpha[0])
    var_1 = np.percentile(pf_sim, alpha[1])
    return var_5, var_1

#### Pipeline - aggregated

In [ ]:
def main():
    